In [1]:
# Color Card Detection and Classification using YOLOv8 OBB
# Detects and classifies color cards with 4-point segmentation
# Classes: 24_color_card, 8_hybrid_card, checker_card

import os
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from ultralytics import YOLO
import yaml

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Ultralytics version: {YOLO.__version__ if hasattr(YOLO, '__version__') else 'installed'}")

Libraries imported successfully!
PyTorch version: 2.8.0+cu126
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4090
Ultralytics version: installed


In [2]:
# Configuration
# Note: Paths are relative to the notebook's directory (notebooks/)
DATA_DIR = Path("dataset/color_card_data")
DATA_YAML = DATA_DIR / "data.yaml"
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Temporary directory for training artifacts (will be cleaned up)
TRAIN_DIR = Path("runs")  # YOLOv8 default training directory

# Training parameters
MODEL_SIZE = "m"  # Options: n (nano), s (small), m (medium), l (large), x (xlarge)
EPOCHS = 200
IMGSZ = (960, 640)
BATCH_SIZE = 16
PATIENCE = 50  # Early stopping patience
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Data directory: {DATA_DIR}")
print(f"Data YAML: {DATA_YAML}")
print(f"Model directory: {MODEL_DIR}")
print(f"Device: {DEVICE}")
print(f"Model size: YOLOv8{MODEL_SIZE}")
print(f"Training epochs: {EPOCHS}")
print(f"Image size: {IMGSZ}")
print(f"Batch size: {BATCH_SIZE}")



Data directory: dataset\color_card_data
Data YAML: dataset\color_card_data\data.yaml
Model directory: models
Device: cuda
Model size: YOLOv8m
Training epochs: 200
Image size: (960, 640)
Batch size: 16


In [3]:
# Verify dataset structure and data.yaml
print("Verifying dataset structure...")

# Check data.yaml
if DATA_YAML.exists():
    with open(DATA_YAML, 'r') as f:
        data_config = yaml.safe_load(f)
    print(f"✓ data.yaml found")
    print(f"  Classes: {data_config['nc']}")
    print(f"  Class names: {data_config['names']}")
else:
    raise FileNotFoundError(f"data.yaml not found at {DATA_YAML}")

# Check directories
train_dir = DATA_DIR / "train" / "images"
val_dir = DATA_DIR / "val" / "images"
test_dir = DATA_DIR / "test" / "images"

train_count = len(list(train_dir.glob("*.jpg"))) if train_dir.exists() else 0
val_count = len(list(val_dir.glob("*.jpg"))) if val_dir.exists() else 0
test_count = len(list(test_dir.glob("*.jpg"))) if test_dir.exists() else 0

print(f"✓ Train images: {train_count}")
print(f"✓ Val images: {val_count}")
print(f"✓ Test images: {test_count}")

# Verify label files
train_labels = DATA_DIR / "train" / "labels"
val_labels = DATA_DIR / "val" / "labels"
test_labels = DATA_DIR / "test" / "labels"

train_label_count = len(list(train_labels.glob("*.txt"))) if train_labels.exists() else 0
val_label_count = len(list(val_labels.glob("*.txt"))) if val_labels.exists() else 0
test_label_count = len(list(test_labels.glob("*.txt"))) if test_labels.exists() else 0

print(f"✓ Train labels: {train_label_count}")
print(f"✓ Val labels: {val_label_count}")
print(f"✓ Test labels: {test_label_count}")

if train_count == 0 or val_count == 0:
    raise ValueError("Dataset split not found. Please run prepare_color_card_dataset.py first.")



Verifying dataset structure...
✓ data.yaml found
  Classes: 3
  Class names: ['24_color_card', '8_hybrid_card', 'checker_card']
✓ Train images: 1008
✓ Val images: 126
✓ Test images: 126
✓ Train labels: 1008
✓ Val labels: 126
✓ Test labels: 126


In [4]:
# Sample annotation verification
# Check a few annotation files to verify OBB format (4 points)
sample_label_file = list(train_labels.glob("*.txt"))[0]
print(f"Sample label file: {sample_label_file.name}")
with open(sample_label_file, 'r') as f:
    lines = f.readlines()
    print(f"Number of annotations: {len(lines)}")
    for i, line in enumerate(lines[:2]):  # Show first 2 annotations
        parts = line.strip().split()
        if len(parts) >= 9:
            class_id = parts[0]
            points = [float(x) for x in parts[1:9]]
            print(f"  Annotation {i+1}: Class {class_id}, Points: {points}")
            print(f"    Format: class x1 y1 x2 y2 x3 y3 x4 y4 (normalized)")


Sample label file: 1-1-3000_jpg.rf.4f8ce991203fed15ffcebd285eafad10.txt
Number of annotations: 1
  Annotation 1: Class 0, Points: [0.0222233333, 0.1393769449, 0.3937033333, 0.1393769449, 0.3937033333, 0.9720403709, 0.0222233333, 0.9720403709]
    Format: class x1 y1 x2 y2 x3 y3 x4 y4 (normalized)


In [5]:
# Initialize YOLOv8 OBB model
# For OBB (Oriented Bounding Box) mode, we use yolov8*-obb.pt models
model_name = f"yolov8{MODEL_SIZE}-obb.pt"
print(f"Initializing model: {model_name}")

# Initialize model in OBB mode
model = YOLO(model_name)

print(f"✓ Model initialized: {model_name}")
print(f"  Model type: OBB (Oriented Bounding Box)")
print(f"  Input size: {IMGSZ}x{IMGSZ}")



Initializing model: yolov8m-obb.pt
✓ Model initialized: yolov8m-obb.pt
  Model type: OBB (Oriented Bounding Box)
  Input size: (960, 640)x(960, 640)


In [6]:
# Train the model
# Training artifacts will be saved to runs/ directory (temporary)
print("Starting training...")
print(f"Dataset config: {DATA_YAML}")
print(f"Training artifacts will be saved to: {TRAIN_DIR}")

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    device=DEVICE,
    patience=PATIENCE,
    save=True,
    save_period=10,  # Save checkpoint every 10 epochs
    project=str(TRAIN_DIR),  # Save to runs/ instead of models/
    name="color_card_obb",
    exist_ok=True,
)

print("Training completed!")
print(f"Best model saved to: {results.save_dir}")



Starting training...
Dataset config: dataset\color_card_data\data.yaml
Training artifacts will be saved to: runs
Ultralytics 8.3.240  Python-3.12.5 torch-2.8.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset\color_card_data\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=(960, 640), int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name

In [7]:
# Load the best model from training and save only the final model to models/
best_model_path = Path(results.save_dir) / "weights" / "best.pt"
print(f"Loading best model from: {best_model_path}")

if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    print("✓ Best model loaded successfully")
else:
    print("⚠ Best model not found, using last checkpoint")
    last_model_path = Path(results.save_dir) / "weights" / "last.pt"
    best_model = YOLO(str(last_model_path))
    best_model_path = last_model_path

# Copy only the final model to models directory (for inference)
final_model_path = MODEL_DIR / f"color_card_yolov8{MODEL_SIZE}_obb.pt"
import shutil
shutil.copy2(best_model_path, final_model_path)
print(f"✓ Final model saved to: {final_model_path}")
print(f"  (Training artifacts remain in: {results.save_dir})")



Loading best model from: G:\GitHub\ascota\notebooks\runs\color_card_obb\weights\best.pt
✓ Best model loaded successfully
✓ Final model saved to: models\color_card_yolov8m_obb.pt
  (Training artifacts remain in: G:\GitHub\ascota\notebooks\runs\color_card_obb)


In [8]:
# Evaluate on validation set
print("Evaluating on validation set...")
val_metrics = best_model.val(data=str(DATA_YAML), split='val')
print("\nValidation Metrics:")
print(f"  mAP50: {val_metrics.box.map50:.4f}")
print(f"  mAP50-95: {val_metrics.box.map:.4f}")
print(f"  Precision: {val_metrics.box.mp:.4f}")
print(f"  Recall: {val_metrics.box.mr:.4f}")


Evaluating on validation set...
Ultralytics 8.3.240  Python-3.12.5 torch-2.8.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
YOLOv8m-obb summary (fused): 101 layers, 26,401,804 parameters, 0 gradients, 80.8 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2310.6320.5 MB/s, size: 225.0 KB)
val: Scanning G:\GitHub\ascota\notebooks\dataset\color_card_data\val\labels.cache... 126 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 126/126  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 3.7it/s 2.2s0.2ss
                   all        126        164      0.998          1      0.995      0.995
         24_color_card         46         46      0.997          1      0.995      0.995
         8_hybrid_card         76         76      0.998          1      0.995      0.995
          checker_card         42         42          1          1      0.995      0.994
Speed: 1.6ms preprocess, 4.8ms inference, 0.0ms loss, 2.9ms 

In [9]:
# Evaluate on test set
print("Evaluating on test set...")
test_metrics = best_model.val(data=str(DATA_YAML), split='test')
print("\nTest Set Metrics:")
print(f"  mAP50: {test_metrics.box.map50:.4f}")
print(f"  mAP50-95: {test_metrics.box.map:.4f}")
print(f"  Precision: {test_metrics.box.mp:.4f}")
print(f"  Recall: {test_metrics.box.mr:.4f}")


Evaluating on test set...
Ultralytics 8.3.240  Python-3.12.5 torch-2.8.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 407.9259.1 MB/s, size: 188.6 KB)
val: Scanning G:\GitHub\ascota\notebooks\dataset\color_card_data\test\labels... 126 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 126/126 2.9Kit/s 0.0s
val: New cache created: G:\GitHub\ascota\notebooks\dataset\color_card_data\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 3.8it/s 2.1s0.2ss
                   all        126        158      0.959          1      0.979      0.978
         24_color_card         41         41      0.953          1      0.994      0.994
         8_hybrid_card         79         79      0.973          1      0.994      0.994
          checker_card         38         38       0.95          1      0.948      0.945
Speed: 1.5ms preprocess, 4.6ms inference, 0.0ms loss, 2.8ms 

In [10]:
# Generate confusion matrix
print("Generating confusion matrix...")
best_model.val(data=str(DATA_YAML), split='test', plots=True, save_json=True)

# The confusion matrix will be saved in the results directory
confusion_matrix_path = Path(results.save_dir) / "confusion_matrix.png"
if confusion_matrix_path.exists():
    print(f"✓ Confusion matrix saved to: {confusion_matrix_path}")


Generating confusion matrix...
Ultralytics 8.3.240  Python-3.12.5 torch-2.8.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 2199.6202.7 MB/s, size: 213.4 KB)
val: Scanning G:\GitHub\ascota\notebooks\dataset\color_card_data\test\labels.cache... 126 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 126/126  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 3.6it/s 2.2s0.2ss
                   all        126        158      0.959          1      0.979      0.978
         24_color_card         41         41      0.953          1      0.994      0.994
         8_hybrid_card         79         79      0.973          1      0.994      0.994
          checker_card         38         38       0.95          1      0.948      0.945
Speed: 1.4ms preprocess, 4.6ms inference, 0.0ms loss, 2.8ms postprocess per image
Saving G:\GitHub\ascota\runs\obb\val3\predictions.json...
Results s

In [11]:
# Visualize predictions on sample test images
print("Generating prediction visualizations...")

test_images = list(test_dir.glob("*.jpg"))[:10]  # Sample 10 test images
print(f"Visualizing predictions on {len(test_images)} test images...")

for img_path in test_images:
    # Run prediction (save to temporary runs/ directory, not models/)
    results_pred = best_model.predict(
        source=str(img_path),
        imgsz=IMGSZ,
        conf=0.25,  # Confidence threshold
        iou=0.45,   # IoU threshold for NMS
        save=True,
        save_txt=False,
        save_conf=True,
        project=str(TRAIN_DIR),
        name="predictions",
        exist_ok=True
    )

print(f"✓ Predictions saved to: {TRAIN_DIR / 'predictions'}")



Generating prediction visualizations...
Visualizing predictions on 10 test images...

image 1/1 g:\GitHub\ascota\notebooks\dataset\color_card_data\test\images\100-1-3000_jpg.rf.2449dbd8fde92394fa1f96a8f5be1143.jpg: 448x640 1 8_hybrid_card, 37.2ms
Speed: 1.5ms preprocess, 37.2ms inference, 2.9ms postprocess per image at shape (1, 3, 448, 640)
Results saved to G:\GitHub\ascota\notebooks\runs\predictions

image 1/1 g:\GitHub\ascota\notebooks\dataset\color_card_data\test\images\104-1-3000_jpg.rf.d232a80ba4fdb061909a7fc5b592b0f6.jpg: 448x640 1 8_hybrid_card, 6.9ms
Speed: 1.3ms preprocess, 6.9ms inference, 1.8ms postprocess per image at shape (1, 3, 448, 640)
Results saved to G:\GitHub\ascota\notebooks\runs\predictions

image 1/1 g:\GitHub\ascota\notebooks\dataset\color_card_data\test\images\107-1-3000_jpg.rf.1c6fdf9afbb1dcdd3b1f4a9b8055fe65.jpg: 448x640 1 8_hybrid_card, 6.6ms
Speed: 1.1ms preprocess, 6.6ms inference, 1.9ms postprocess per image at shape (1, 3, 448, 640)
Results saved to G:\

In [12]:
# Test on images with multiple cards (if available)
# Find images with 2 cards by checking label files with 2+ lines
multi_card_images = []
for label_file in test_labels.glob("*.txt"):
    with open(label_file, 'r') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
        if len(lines) >= 2:
            img_name = label_file.stem + ".jpg"
            img_path = test_dir / img_name
            if img_path.exists():
                multi_card_images.append(img_path)

if multi_card_images:
    print(f"Found {len(multi_card_images)} test images with multiple cards")
    print("Testing on multi-card images...")
    
    sample_multi = multi_card_images[0]
    results_multi = best_model.predict(
        source=str(sample_multi),
        imgsz=IMGSZ,
        conf=0.25,
        iou=0.45,
        save=True,
        project=str(TRAIN_DIR),  # Save to runs/, not models/
        name="multi_card_predictions",
        exist_ok=True
    )
    print(f"✓ Multi-card prediction saved to: {TRAIN_DIR / 'multi_card_predictions'}")
else:
    print("No multi-card images found in test set")


Found 32 test images with multiple cards
Testing on multi-card images...

image 1/1 g:\GitHub\ascota\notebooks\dataset\color_card_data\test\images\14-1-3000_jpg.rf.34beaedec8f2bb9782dd0c008149a564.jpg: 448x640 1 24_color_card, 1 checker_card, 10.1ms
Speed: 1.8ms preprocess, 10.1ms inference, 2.2ms postprocess per image at shape (1, 3, 448, 640)
Results saved to G:\GitHub\ascota\notebooks\runs\multi_card_predictions
✓ Multi-card prediction saved to: runs\multi_card_predictions


In [ ]:
# Export to ONNX (for cross-platform deployment)
try:
    onnx_path = best_model.export(format='onnx', imgsz=IMGSZ)
    # Move ONNX to models directory
    onnx_final = MODEL_DIR / f"color_card_yolov8{MODEL_SIZE}_obb.onnx"
    import shutil
    shutil.move(onnx_path, onnx_final)
    print(f"✓ ONNX model saved to: {onnx_final}")
except Exception as e:
    print(f"⚠ ONNX export failed: {e}")

In [ ]:
# Export to TorchScript
try:
    torchscript_path = best_model.export(format='torchscript', imgsz=IMGSZ)
    # Move TorchScript to models directory
    torchscript_final = MODEL_DIR / f"color_card_yolov8{MODEL_SIZE}_obb.torchscript"
    import shutil
    shutil.move(torchscript_path, torchscript_final)
    print(f"✓ TorchScript model saved to: {torchscript_final}")
except Exception as e:
    print(f"⚠ TorchScript export failed: {e}")

In [ ]:
print(f"\n✓ Final inference models saved to {MODEL_DIR}:")
print(f"  - PyTorch: {final_model_path.name}")
if 'onnx_final' in locals() and onnx_final.exists():
    print(f"  - ONNX: {onnx_final.name}")
if 'torchscript_final' in locals() and torchscript_final.exists():
    print(f"  - TorchScript: {torchscript_final.name}")
    

In [13]:
# Example: Using the trained model for inference
print("Example inference usage:")

# Example: Run prediction on a test image
sample_image = list(test_dir.glob("*.jpg"))[0]
print(f"Running inference on: {sample_image.name}")

results = best_model.predict(
    source=str(sample_image),
    imgsz=IMGSZ,
    conf=0.25,
    iou=0.45,
    verbose=False
)

# Extract detection information
detections = []
for result in results:
    if result.obb is not None:
        for i, (cls, conf) in enumerate(zip(result.obb.cls, result.obb.conf)):
            # Get 4 corner points
            points = result.obb.xyxyxyxy[i].cpu().numpy()  # Shape: (4, 2)
            # Normalize points (they're in pixel coordinates)
            img_h, img_w = result.orig_shape
            points_normalized = points / np.array([img_w, img_h])
            
            detection = {
                'class_id': int(cls.item()),
                'class_name': best_model.names[int(cls.item())],
                'confidence': float(conf.item()),
                'points_normalized': points_normalized.tolist(),
                'points_pixel': points.tolist()
            }
            detections.append(detection)

# Display results
print(f"\nFound {len(detections)} color card(s):")
for i, det in enumerate(detections, 1):
    print(f"\n  Detection {i}:")
    print(f"    Class: {det['class_name']} (ID: {det['class_id']})")
    print(f"    Confidence: {det['confidence']:.4f}")
    print(f"    4 Corner Points (normalized): {det['points_normalized']}")
    print(f"    4 Corner Points (pixel): {det['points_pixel']}")

print(f"\n✓ Inference example completed")


Example inference usage:
Running inference on: 100-1-3000_jpg.rf.2449dbd8fde92394fa1f96a8f5be1143.jpg

Found 1 color card(s):

  Detection 1:
    Class: 8_hybrid_card (ID: 1)
    Confidence: 0.9751
    4 Corner Points (normalized): [[0.015074998140335083, 0.9698708842823481], [0.3613582253456116, 0.9695565040778276], [0.3610314130783081, 0.16024059781567418], [0.014748185873031616, 0.16055491099769514]]
    4 Corner Points (pixel): [[30.87359619140625, 1324.8436279296875], [740.0616455078125, 1324.4141845703125], [739.392333984375, 218.88865661621094], [30.20428466796875, 219.31800842285156]]

✓ Inference example completed
